In [1]:
import pandas as pd
import datetime
from datetime import date, timedelta
from datetime import datetime
import webbrowser
import psutil
import os
import shutil
import subprocess
import openpyxl
import win32com.client
import datetime
import nbformat
import pathlib
from nbconvert import PythonExporter

In [2]:
notebook_dict = {
  'IEX.ipynb': 1,
  'RTA.ipynb': 1,
}

def convert_notebook(notebook_path):
    exporter = PythonExporter()
    output, _ = exporter.from_filename(notebook_path)
    return output

first_glob_1 = "C:/Users/huuchinh.nguyen"
first_glob_2 = "C:/Users/ADMIN"
first_glob_3 = "C:/Users/hoangminhduc.nguyen"
first_glob_4 = "C:/Users/duonghoangvu.pham"


if os.path.exists(first_glob_1):
    first_glob = first_glob_1
elif os.path.exists(first_glob_2):
    first_glob = first_glob_2
elif os.path.exists(first_glob_3):
    first_glob = first_glob_3
elif os.path.exists(first_glob_4):
    first_glob = first_glob_4
else:
    raise FileNotFoundError(f"Neither {first_glob_1} nor {first_glob_2} nor {first_glob_3} nor {first_glob_4} exists.")

In [3]:


for nbook in notebook_dict:
    if notebook_dict[nbook] == 0:
        print(f"Not executing notebook {nbook}.")
        continue
        
    nbook_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/{nbook}'
    
    if os.path.isfile(nbook_path):
        print(f"Running notebook: {nbook}")
        try:
            script = convert_notebook(nbook_path)
            exec(script)
        except Exception as e:
            print(f"Error running notebook: {nbook}")
            print(str(e))
    else:
        print(f"Error: Notebook {nbook} not found.")

Running notebook: IEX.ipynb


<string>:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
<string>:171: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
<string>:180: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
<string>:186: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:192: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:193: UserWarning: Could not infer format, so each element w

Running notebook: RTA.ipynb


<string>:126: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
<string>:225: DeprecationWarning: `DataFrame.melt` is deprecated. Use `unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`


shape: (5, 4)
┌────────┬───────────┬──────────────────────────┬────────────────────┐
│ Month  ┆ EID       ┆ employee_name            ┆ count_working_days │
│ ---    ┆ ---       ┆ ---                      ┆ ---                │
│ str    ┆ i64       ┆ str                      ┆ i32                │
╞════════╪═══════════╪══════════════════════════╪════════════════════╡
│ Apr-25 ┆ 101993774 ┆ HIEU THI TRAN            ┆ 22                 │
│ Apr-25 ┆ 102029874 ┆ THIEN THANH TOAN  TRUONG ┆ 20                 │
│ Apr-25 ┆ 102173363 ┆ THI NGOC BICH  TRAN      ┆ 23                 │
│ Apr-25 ┆ 102173384 ┆ HUU VINH  MAI            ┆ 22                 │
│ Apr-25 ┆ 102211178 ┆ THI QUYNH TRANG  NGUYEN  ┆ 22                 │
└────────┴───────────┴──────────────────────────┴────────────────────┘
[None, 'After Call Work', 'Available Chat', 'Available Phone', 'Break', 'Coaching', 'End of Shift', 'Login', 'Lunch', 'Not Ready', 'Offline', 'Outbound Call', 'Phone Talking', 'Restroom', 'System Issue', '

In [4]:
import polars as pl
import glob

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8")
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True)
        
        df = df.with_columns([
            pl.col(col).cast(pl.String) 
            for col in df.columns
        ])
        
        df_list.append(df)
    
    merged_df = pl.concat(df_list, how='vertical')

    return merged_df

In [5]:
RTA_REPORT_GENERATE = input_data(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA')
print(RTA_REPORT_GENERATE.columns)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Date_Converted').str.strptime(pl.Date, format='%Y-%m-%d'),
    pl.col('Target').cast(pl.Float64), 
    pl.col('duration').cast(pl.Float64),
    pl.col('sum_productive').cast(pl.Float64),
    pl.col('start').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col('end').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False)
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.filter(pl.col('duration').is_not_null())
RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Shift Tracking').cast(pl.Utf8).str.strip_chars(),
    pl.col('Open Time').cast(pl.Float64).fill_null(0.0), 
    pl.col('Extra Time').cast(pl.Float64).fill_null(0.0)    
])
 
RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.when(pl.col('Shift Tracking') == 'HDL').then(0.5)
    .when(
        (pl.col('Shift Tracking').is_in(['PR','PR - OT','PO'])) & 
        ((pl.col('Open Time') > 0) | (pl.col('Extra Time') > 0))
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking').is_in([
            'Training Offline', 'Sick Leave', 'Paid Leave', 'Billable Training'
        ])) & (pl.col('Open Time') == 0)
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking') != 'HDL') & 
        (pl.col('Shift Tracking') != 'PR') & 
        (pl.col('Open Time') == 0)
    ).then(0.0)
    .otherwise(None)
    .alias('hc_present')
])
 
filtered_data = RTA_REPORT_GENERATE.select([
    'Date_Converted', 'Employee Name', 'Detail Status','IEX ID','First Shift','Shift Tracking', 
    'start', 'end','Target', 'duration', 'sum_productive','training-idle','Extra Time', 'hc_schedule',
    'hc_actual', 'hc_present'
])
filtered_data = filtered_data.with_columns([
    pl.col('hc_actual').cast(pl.Float64),
    pl.col('hc_present').cast(pl.Float64)
])
 
start_date = '2026-04-01'
end_date = '2026-04-16'
 
start_date_parsed = pl.lit(start_date).str.strptime(pl.Date, format='%Y-%m-%d')
end_date_parsed = pl.lit(end_date).str.strptime(pl.Date, format='%Y-%m-%d')
 
sorted_data = filtered_data.sort('Date_Converted')
filtered_data = sorted_data.filter((pl.col('Date_Converted') >= start_date_parsed) & (pl.col('Date_Converted') <= end_date_parsed))
 
filtered_data_pd = filtered_data.to_pandas()

['Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id', 'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled', 'Night_Shift', 'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift', 'start', 'end', 'total_time_chat_handle', 'sum_productive', 'break', 'lunch', 'coaching-idle', 'training-idle', 'outbound-idle', 'break_count', 'other_status', 'over_break', 'over_lunch', 'exceed_break', 'duration', 'hc_actual', 'hc_schedule', 'time_late', 'time_leave', 'lateness', 'adherence_time', 'total_time_pull_out', 'ramco_marked', 'nsa_authorize', 'ot_authorize', 'hours', 'ot_

In [6]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(100)

def filter_shift_discrepancies(df_input) -> pl.DataFrame:
    """
    Filters the dataframe to identify discrepancies in headcount and shifts.
    Ensures column names match the provided schema (Case-Sensitive).
    """
    # Convert to Polars if input is Pandas
    if isinstance(df_input, pd.DataFrame):
        df = pl.from_pandas(df_input)
    else:
        df = df_input

    # Define filter conditions using exact column names from your error log
    # Requirement: Target > 0 but hc_actual == 0
    target_gap_condition = (
        (pl.col("Target").cast(pl.Float64, strict=False).fill_null(0.0) > 0) & 
        (pl.col("hc_actual").cast(pl.Float64, strict=False).fill_null(0.0) == 0)
    )

    # Requirement: hc_actual != hc_present
    hc_mismatch_condition = (
        pl.col("hc_actual").cast(pl.Float64, strict=False).fill_null(0.0) != 
        pl.col("hc_present").cast(pl.Float64, strict=False).fill_null(0.0)
    )

    # Main filtering logic
    result = df.filter(
        (hc_mismatch_condition | target_gap_condition) & 
        (pl.col("First Shift").str.contains("-")) & 
        (pl.col("Employee Name").is_not_null()) &
        (pl.col("Shift Tracking") != "PO")
    ).sort("Date_Converted")

    return result

# Execute with the correct variable
# Assuming 'filtered_data' is your source data
filtered_data_1 = filter_shift_discrepancies(filtered_data)
filtered_data_1

Date_Converted,Employee Name,Detail Status,IEX ID,First Shift,Shift Tracking,start,end,Target,duration,sum_productive,training-idle,Extra Time,hc_schedule,hc_actual,hc_present
date,str,str,str,str,str,datetime[ms],datetime[ms],f64,f64,f64,str,f64,str,f64,f64


In [7]:
filtered_data = filtered_data.with_columns([
    pl.col("Date_Converted").cast(pl.Date)
])

filtered_data_3 = filtered_data.filter(
    (pl.col("Date_Converted") >= date(2026, 4, 1)) & 
    (pl.col("Date_Converted") <= date(2026, 4, 27)) & 
    (pl.col("Shift Tracking").is_in(["PO", "PR - OT"]))
)

filtered_data_3

Date_Converted,Employee Name,Detail Status,IEX ID,First Shift,Shift Tracking,start,end,Target,duration,sum_productive,training-idle,Extra Time,hc_schedule,hc_actual,hc_present
date,str,str,str,str,str,datetime[ms],datetime[ms],f64,f64,f64,str,f64,str,f64,f64


In [8]:
import pathlib

def get_xlsx_filenames(data_dir):
    xlsx_files = []
    for filename in pathlib.Path(data_dir).glob('**/*.xlsx'):
        xlsx_files.append(filename.name)
    return xlsx_files
def create_dataframe_with_filenames(data_dir):
    xlsx_filenames = get_xlsx_filenames(data_dir)
    df = pd.DataFrame(xlsx_filenames, columns=['Filename'])
    return df
def get_latest_xlsx_files(df):
    df = df[df['Filename'].str.match(r'\d{4}-\d{2}-\d{2}\.xlsx')]
    df['Date'] = pd.to_datetime(df['Filename'].str.replace('.xlsx', ''), format='%Y-%m-%d')
    df_sorted = df.sort_values(by='Date', ascending=False)
    latest_files = df_sorted.head(8)['Filename'].tolist()
    return latest_files
def delete_all_xlsx_files(directory):
    for filename in pathlib.Path(directory).glob('*.xlsx'):
        os.remove(filename)
def copy_latest_files(source_dir, dest_dir, latest_files):
    for filename in latest_files:
        source_path = pathlib.Path(source_dir) / filename
        dest_path = pathlib.Path(dest_dir) / filename
        shutil.copy(source_path, dest_path)

source_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA'
dest_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_RTA_FOR_REPORT'

df = create_dataframe_with_filenames(source_dir)
latest_files = get_latest_xlsx_files(df)
delete_all_xlsx_files(dest_dir)
copy_latest_files(source_dir, dest_dir, latest_files)

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_27956\3116803128.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Filename'].str.replace('.xlsx', ''), format='%Y-%m-%d')
